Hands-On Lab: End-to-End Mini-Project   
● Step 1: Choose a provided dataset and determine whether the task is regression or classification.     
● Step 2: Perform brief EDA, then preprocess: handle missing values, encode categoricals, and scale features 
(fit on train only).    
● Step 3: Train at least two appropriate models and evaluate them with a suitable metric against a baseline.    
● Step 4: Select the better model, justify the choice, and document the full pipeline in a narrated notebook.   
● Step 5: Commit the finished mini-project notebook to GitHub with a clear commit message.  

Import important libraries:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

loading & cleaning dataset:

In [ ]:
df = pd.read_csv('heart_disease_risk_2026.csv') # load the dataset

In [ ]:
print(df.head(5))

looking at a sample of our data, i think it is a categiorical issue. since we need to specify whether there is heart desise risk(%) or not.------------(review this after work :) )--------------

In [ ]:
print("shape: ",df.shape,"\n")

print("duplicates: ",df.duplicated().sum() ,"\n") # check for duplicates

print("null values : ",df.isnull().sum() ,"\n") 

The dataset has no null nor duplicated values.

Data has 9000 rows - 27 columns

In [ ]:
df.describe().T # .T is transpose, it will convert rows to columns and columns to rows

EDA:

Lets check who has heart disease & visualize it :

In [ ]:
print("Percentage of people with heart disease: ", df['has_heart_disease'].mean() * 100, "%") # the mean gives us the percentage of people who have heart disease

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x='has_heart_disease', data=df, ax=axes[0])

axes[0].set_title('Count of Heart Disease')

sns.histplot(x='age', hue='has_heart_disease', data=df, ax=axes[1], bins=20, kde=True) # kde=True adds a kernel density estimate to the histogram. hue='has_heart_disease' will color the histogram based on whether the person has heart disease or not.
                                                                                       # bins=20 will divide the age range into 20 equal parts and plot the histogram accordingly.
axes[1].set_title('Age Distribution by Heart Disease Status')


from the above graph we can see a visualization that demonstrates The age which linked to the most heart diseases.

The 'Two cholesterols'  and their relation to heart disease: (HDL = good, LDL= bad)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(x='ldl', hue='has_heart_disease', data=df, ax=axes[0], bins=20, kde=True) # hue adds has heart disease on top of the base ldl and change color 
axes[0].set_title('LDL -bad- Distribution by Heart Disease Status')

sns.histplot(x='hdl', hue='has_heart_disease', data=df, ax=axes[1], bins=20, kde=True)
axes[1].set_title('HDL -good- Distribution by Heart Disease Status')


From the above figure we can clearly see:   
  positive correlation between high LDL and heart disease.   
  negative correlation between high HDL and heart disease.

sleep and heart disease:

different way of visualizing,  
 An explanation of the following process:   
   1- Bins continuous sleep-hour values into 'predefined categorical' intervals using pd.cut().   
   2- Computes the mean binary heart-disease outcome per interval and converts it to a percentage.(Sleep_Risk).   
   3- Plots interval-level as a line chart with categorical x-axis labels.  
   4- Adds the dataset-wide prevalence (haert_dis_per) as a horizontal reference line for comparison. 


In [ ]:


fig, axes = plt.subplots(1, 1, figsize=(12, 7))

bins = [3, 5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 11] 

sleep_labels = ['3–5' , '5–6' , '6–6.5' , '6.5–7' , '7–7.5' , '7.5–8' , '8–8.5' , '8.5–9' , '9–11' ]

# Group sleep hours into labelled intervals
df['sleep_level'] = pd.cut( df['sleep_hours'], bins=bins, labels=sleep_labels, include_lowest=True, right=True )



sleep_risk = df.groupby('sleep_level', observed=True)['has_heart_disease'].mean().mul(100) 
  # observed=True will only consider the sleep_level that are present in the dataset. If we don't use observed=True, it will consider all the possible values of sleep_hours and fill the missing values with NaN.
  # mean() will give us the percentage of people with heart disease for each sleep_level value.
  # We can then plot this data using a line plot or a bar plot. Here, we will use a line plot.

heart_dis_per = df['has_heart_disease'].mean()*100  


axes.plot( sleep_risk.index.astype(str) , sleep_risk.values , marker='o', color="#751F6E")

axes.set_title('Heart Disease Risk by Sleep Hours')

axes.set_ylabel('Heart disease risk (%)')

axes.axhline (y = heart_dis_per, color = 'black', ls = '--', label = 'Percentage of people with heart disease')

axes.legend()

axes.grid(axis='y', alpha=0.3)


Pair plot : 

In [ ]:
sns.pairplot(df) 

The Pair plot is too big and unclear, it is not really that usefull in this case.

Lets explore some "ALL" common-thought correlations:

In [ ]:
corr = (df.drop(columns=['patient_id'], errors='ignore').corr(numeric_only=True)) # correlation for the dataset but we drop the patient id

mask = np.triu(np.ones_like(corr, dtype=bool)) # show only the bottom half

plt.figure(figsize=(16, 12))

sns.heatmap(corr,
    mask=mask,
    annot=True,
    fmt='.2f',             
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    annot_kws={'size': 8},
    cbar_kws={'shrink': 0.8}
)

plt.title('Correlation Matrix', fontsize=16)
plt.xticks(rotation=45, ha='right') # rotates the X labels so they become readable
plt.tight_layout()
plt.show()

we have some interesting strong correlations:   

1- Fasting blood sugar with hba1c :    0.88  
    higher fasting blood sugar tends to occur with higher HbA1c.    

2- Total cholestrol and LDL :   0.84    
    it is a positive correlation because LDL contributes substantially to total cholesterol.  

3- Systolic and diastolic blood pressure:   0.77    
    People with higher systolic blood pressure also tend to have higher diastolic blood pressure.

4- Age and maximum heart rate achieved:  -0.73   
    This is a strong negative correlation, the more the age the less the achivable maximum heart rate.  

5- Exercise-induced angina and having a heart disease:  0.45    
    having an angina induced by exercise is positivly related to having a heart disease.




"Exercise-induced angina: chest pain or discomfort that happens when your heart works harder during physical activity and does not get enough oxygen-rich blood"

How "each" correlates with "heart disease":

In [ ]:
numeric = df.drop(columns = ['patient_id'])

corel = numeric.corr(numeric_only=True)['has_heart_disease'].drop(['has_heart_disease']).sort_values()

''' 
-numeric.corr(numeric_only=True): Calculates the correlation between every pair of numeric columns

-['has_heart_disease']: Selects only the correlations between each numeric variable and has_heart_disease.

-.drop(df['has_heart_disease']): Removes the correlation of has_heart_disease with itself, which is always 1.0.

'''

colors = ['red' if value > 0 else 'blue' for value in corel]

plt.figure(figsize=(12, 6))

ax = corel.plot(kind= 'barh', color = colors ) # barh = horizantal bar
ax.set_xlabel("Correlation amount")
ax.set_title("correlation with heart disease risk (Red increase , blue decrease)")
plt.tight_layout()
plt.show()

Classification model training:

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report, confusion_matrix , roc_auc_score, RocCurveDisplay, accuracy_score, roc_curve


from sklearn.dummy import DummyClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

Load data & split it :

In [ ]:
heart_data = df.drop(columns=['has_heart_disease', 'patient_id'])

''' 
pd.getdummy() performs one-hot encoding.
(converts categorical columns in heart_data into numerical columns)
drop_first=True: removes the first category from each encoded column to avoid redundant information.
'''

X = pd.get_dummies(heart_data, drop_first = True) # features
y = df['has_heart_disease'] # target 

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size= .30 ,random_state=42)

Scale data, fit , predict:

In [ ]:
BL = make_pipeline(StandardScaler(), DummyClassifier(strategy= 'most_frequent'))
LR = make_pipeline(StandardScaler(),LogisticRegression())
RF = make_pipeline(StandardScaler(), RandomForestClassifier())
SVM = make_pipeline(StandardScaler(), SVC())
KNN = make_pipeline(StandardScaler(), KNeighborsClassifier())

BL.fit(X_train, y_train)
LR.fit(X_train, y_train)
RF.fit(X_train, y_train)
SVM.fit(X_train, y_train)
KNN.fit(X_train, y_train)

BL_pred = BL.predict(X_test)
LR_pred = LR.predict(X_test)
RF_pred = RF.predict(X_test)
SVM_pred = SVM.predict(X_test)
KNN_pred = KNN.predict(X_test)


Calculate Accuracy, confusion matrix, classification report & ROC:

0 = disease, 1 = no disease---------------
That is why  probabilities[:,1] is used

In [ ]:
BL_acc  = accuracy_score(y_test, BL_pred)
LR_acc  = accuracy_score(y_test, LR_pred)
RF_acc  = accuracy_score(y_test, RF_pred)
SVM_acc = accuracy_score(y_test, SVM_pred)
KNN_acc = accuracy_score(y_test, KNN_pred)

BL_cm  = confusion_matrix(y_test , BL_pred)
LR_cm  = confusion_matrix(y_test , LR_pred)
RF_cm  = confusion_matrix(y_test, RF_pred)
SVM_cm = confusion_matrix(y_test, SVM_pred)
KNN_cm = confusion_matrix(y_test, KNN_pred)

BL_Class_Rep = classification_report(y_test, BL_pred , target_names=['Disease', 'No Disease'])
LR_Class_Rep = classification_report(y_test, LR_pred , target_names=['Disease', 'No Disease'])
RF_Class_Rep = classification_report(y_test, RF_pred , target_names=['Disease', 'No Disease'])
SVM_Class_Rep = classification_report(y_test, SVM_pred , target_names=['Disease', 'No Disease'])
KNN_Class_Rep = classification_report(y_test, KNN_pred , target_names=['Disease', 'No Disease'])

BL_probabilities = BL.predict_proba(X_test)
LR_probabilities = LR.predict_proba(X_test)
RF_probabilities = RF.predict_proba(X_test)
SVM_probabilities = SVM.decision_function(X_test)
KNN_probabilities = KNN.predict_proba(X_test)

BL_auc_score = roc_auc_score(y_test, BL_probabilities[:,1])
LR_auc_score = roc_auc_score(y_test, LR_probabilities[:,1])
RF_auc_score = roc_auc_score(y_test, RF_probabilities[:,1])
SVM_auc_score = roc_auc_score(y_test,SVM_probabilities )
KNN_auc_score = roc_auc_score(y_test, KNN_probabilities[:,1])


In [ ]:
print("Classes:", BL.classes_)

Printing resualts:

In [ ]:
print ("Accuracies of models: \n")

print(f"Baseline (most frequent)'s accuracy: {BL_acc}\n")

print(f"Logistic Regression's accuracy: {LR_acc}\n")
print(f"Random Forest's accuracy: {RF_acc}\n")
print(f"Support Vector Machine's accuracy: {SVM_acc}\n")
print(f"K-Nearest Neighbor's accuracy: {KNN_acc}\n")

In [ ]:
print("ROC-AUC Scores:\n")

print(f"Baseline (most frequent)'s ROC-AUC score: {BL_auc_score:.4f}\n")

print(f"Logistic Regression's ROC-AUC score: {LR_auc_score:.4f}\n")
print(f"Random Forest's ROC-AUC score: {RF_auc_score:.4f}\n")
print(f"Support Vector Machine's ROC-AUC score: {SVM_auc_score:.4f}\n")
print(f"K-Nearest Neighbor's ROC-AUC score: {KNN_auc_score:.4f}\n")

In [ ]:
print("Confusion matricies:\n")

print(f"Baseline (most frequent)'s confusion matrix:\n {BL_cm}\n")

print(f"Logistic Regression's confusion matrix:\n {LR_cm}\n")
print(f"Random Forest's confusion matrix:\n {RF_cm}\n")
print(f"Support Vector Machine's confusion matrix:\n {SVM_cm}\n")
print(f"K-Nearest Neighbor's confusion matrix:\n {KNN_cm}\n")


In [ ]:
print ("Classification Reports : \n")

print(f"Baseline (most frequent)'s Classification Report:\n {BL_Class_Rep}\n")


In [ ]:
print ("Classification Reports : \n")


print(f"Logistic Regression's Classification Report:\n {LR_Class_Rep}\n")
print(f"Random Forest's Classification Report:\n {RF_Class_Rep}\n")
print(f"SVM's Classification Report:\n {SVM_Class_Rep}\n")
print(f"KNN's Classification Report:\n {KNN_Class_Rep}\n")

In [ ]:
## THE ROC CURVE WAS DONE USING CHATGPT

fig, ax = plt.subplots(figsize=(9, 7))

models = {
    "Baseline": BL,
    "Logistic Regression": LR,
    "Random Forest": RF,
    "SVM": SVM,
    "KNN": KNN
}

colors = {
    "Baseline": "gray",
    "Logistic Regression": "blue",
    "Random Forest": "green",
    "SVM": "red",
    "KNN": "purple"
}

for name, model in models.items():
    display = RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        ax=ax,
        name=name
    )

    # Change the color of the curve after it is created
    display.line_.set_color(colors[name])

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="black",
    label="Random chance"
)

ax.set_title("ROC Curves of Classification Models")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Model Comparison

Five models were evaluated: the baseline classifier, Logistic Regression, Random Forest, Support Vector Machine, and K-Nearest Neighbors.   

Since the dataset is imbalanced, accuracy alone is not enough to choose the best model. 
 Therefore, ROC-AUC, precision, recall, F1-score, and the confusion matrix were also considered.    

| Model                  |   Accuracy |    ROC-AUC | Weighted F1-score |
| ---------------------- | ---------: | ---------: | ----------------: |
| Baseline               |     0.6941 |     0.5000 |              0.57 |
| Logistic Regression    | **0.9022** | **0.9579** |          **0.90** |
| Random Forest          |     0.8826 |     0.9417 |              0.88 |
| Support Vector Machine |     0.8919 |     0.9489 |              0.89 |
| K-Nearest Neighbors    |     0.8181 |     0.8530 |              0.81 |

### Selected Model

**Logistic Regression was selected as the best model.** 

It achieved the highest accuracy of **90.22%** and the highest ROC-AUC score of **0.9579**. 
 This means that it produced the strongest overall predictions and had the best ability to distinguish between the two classes. 

Logistic Regression also performed well for both classes. It achieved an F1-score of **0.93** for Disease and **0.83** for No Disease.  
 Its weighted-average F1-score was **0.90**, which was the highest among all tested models. 

Its confusion matrix was:

```text
[[1768, 106],
 [ 158, 668]]
```

The model correctly classified 1,768 Disease cases and 668 No Disease cases.    
 It made a total of **264 incorrect predictions**, which was fewer than SVM with 292 errors, Random Forest with 317 errors, and KNN with 491 errors.    

### Justification

The baseline model was not suitable because it predicted every observation as the majority class.   
 Although it achieved an accuracy of 69.41%, its ROC-AUC score was only 0.50, showing that it could not meaningfully distinguish between the classes.   

SVM and Random Forest also performed well, but both had slightly lower accuracy, ROC-AUC, and F1-scores than Logistic Regression.   
 KNN performed considerably worse, especially for the No Disease class. 

Therefore, Logistic Regression was chosen because it provided the best combination of accuracy, class balance, ROC-AUC performance, and interpretability.   
